In [0]:
from pyspark.sql.functions import col, regexp_extract
from pyspark.sql.types import IntegerType

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df_bronze = spark.table("scottish_housing_ws.bronze.crime")

df_bronze.select("featuretype").distinct().show(10, truncate=False)
df_bronze.select("measurement").distinct().show(10, truncate=False)
df_bronze.select("crime_or_offence").distinct().show(50, truncate=False)

In [0]:
aggregate_crime_types = [
    "All Crimes",
    "All Offences",
    "All Group 1: Non-sexual crimes of violence",
    "All Group 2: Sexual crimes",
    "All Group 3: Crimes of dishonesty",
    "All Group 4: Damage and reckless behaviour",
    "All Group 5: Crimes against society",
    "All Group 6: Antisocial offences",
    "All Group 7: Miscellaneous offences",
    "All Group 8: Road traffic offences"
]

df_silver = (
    df_bronze
    .filter(col("measurement") == "Count")
    .filter(col("crime_or_offence").isin(aggregate_crime_types))
    # Extract first 4 digits of datecode as fiscal year start
    .withColumn("fiscal_year_start", regexp_extract(col("datecode"), r"(\d{4})", 1).cast(IntegerType()))
    .withColumnRenamed("featurecode", "area_code")
    .withColumnRenamed("featurename", "area_name")
    .withColumnRenamed("featuretype", "area_type")
    .withColumnRenamed("crime_or_offence", "crime_type")
    .withColumnRenamed("value", "crimes_recorded")
    .drop("measurement", "units", "datecode")
    .select(
        "fiscal_year_start",
        "area_code",
        "area_name",
        "area_type",
        "crime_type",
        "crimes_recorded"
    )
)

df_silver.printSchema()

In [0]:
df_silver.orderBy("fiscal_year_start", "area_name", "crime_type").show(20, truncate=False)

In [0]:
from pyspark.sql.functions import min, max, count
df_silver.agg(
    count("*").alias("row_count"),
    min("fiscal_year_start").alias("earliest"),
    max("fiscal_year_start").alias("latest")
).show()

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.silver.crime")
)

In [0]:
# Recorded crimes and offences by local authority and Scotland total.
# Fiscal year format: fiscal_year_start is the calendar year the fiscal year begins.
# e.g. fiscal_year_start=1996 represents 1996/1997.
# Group-level aggregates only -- subcategories dropped.
# Count measure only -- rates to be derived in gold using population data.
# Covers 1996 to 2023 (1996/97 to 2023/24).